In [ ]:
#header imports and function defs:
from numpy import genfromtxt
import os, sys
import numpy as np
import scipy as sp
import xarray as xr
import pandas as pd
from netCDF4 import Dataset as ncopen 
from eofs.standard import Eof
from astropy.timeseries import LombScargle
import pyleoclim as pyleo


In [ ]:
def calc_Anom(varIn, period=None, doy=None, numYr=None, ops=None, idxClim=None, conv=None, z_max=5, median=False):
    """
    Compute climatology and anomalies. Handles both regularly-spaced data
    (via period) and irregularly-spaced data with an explicit day-of-year
    array (via doy). Time is assumed to be axis 0 on entry (decorator handles this).

    Parameters
    ----------
    varIn   : ndarray
        Input data, time on axis 0.
    period  : int, optional
        Steps per year for regularly-spaced data (e.g. 12 for monthly, 669
        for Mars daily, 365 for Earth daily, 59900//41 for sub-daily Earth data). 
        If provided, doy is constructed automatically.
    doy     : array-like, optional
        Explicit day-of-year array (1-based) for irregularly-spaced data.
        One of period or doy must be provided.
    numYr   : int, optional
        Number of years. Only used with period; inferred if not provided.
    ops     : int, optional
        If provided, also return data averaged to daily resolution by
        averaging every ops steps. Only used with period.
    idxClim : array-like, optional
        Indices into the time axis to use when building the climatology
        (e.g. to exclude dust storm years). All time steps used if None.
    conv    : int, optional
        Gaussian smoothing half-window applied to climatology across the
        doy axis (wrapping) to reduce susceptible to outliers. Skipped if None.
    z_max   : float, optional
        Z-score threshold for outlier removal when building climatology.
        Set to 0 or None to skip. Default 5.
    median  : bool, optional
        If True, use median instead of mean for climatology. Default False.

    Returns
    -------
    varAnom : ndarray
        Anomalies, same shape as varIn (trimmed to numYr*period if period used).
    climVar : ndarray
        Climatology, shape (n_doy, ...).
    errVar  : ndarray
        Standard error of the climatology, shape (n_doy, ...).
    varDaily : ndarray or None
        Daily-averaged data. Only returned if ops is provided.
    varAnomDaily : ndarray or None
        Daily-averaged anomalies. Only returned if ops is provided.
    """

    varIn = np.asarray(varIn, dtype=float)
    spatial_shape = varIn.shape[1:]   # () for 1D, (lat,) for 2D, (lat,lon) for 3D

    # --- Build DOY array from period if not provided explicitly ---
    if doy is None and period is not None:
        nT    = varIn.shape[0]
        numYr = numYr or (nT // period)
        nUse  = numYr * period
        varIn = varIn[:nUse]                          # trim to exact multiple
        doy   = np.tile(np.arange(1, period + 1), numYr)   # 1-based, repeating
    elif doy is not None:
        doy  = np.asarray(doy, dtype=int)
        nUse = len(doy)
        varIn = varIn[:nUse]
    else:
        raise ValueError("One of period or doy must be provided.")

    # --- Climatology index: default to all time steps ---
    if idxClim is None:
        idxClim = np.arange(nUse)
    idxClim = np.asarray(idxClim, dtype=int)

    doyClim  = doy[idxClim]                          # doy values used for climatology
    n_doy    = int(np.nanmax(doyClim))               # number of doy bins

    # --- Allocate outputs ---
    climVar = np.full((n_doy,)    + spatial_shape, np.nan)
    errVar  = np.full((n_doy,)    + spatial_shape, np.nan)
    varAnom = np.full(varIn.shape,                  np.nan)

    # --- Build climatology from only values indicated by idxClim ---
    for ii in range(np.min(np.unique(doyClim)) - 1, n_doy):
        idx = np.flatnonzero(doyClim == ii + 1)     # time indices for this doy
        if len(idx) == 0:
            continue
        slc = varIn[idxClim][idx]                   # shape (n_matches, ...) or (n_matches,)

        if z_max:
            slc = zscoreVar(slc, axis=0, z_max=z_max)

        climVar[ii] = np.nanmedian(slc, axis=0) if median else np.nanmean(slc, axis=0)
        errVar[ii]  = sp.stats.sem(slc, axis=0, nan_policy='omit')

    # --- Optional smoothing of climatology ---
    if conv is not None:
        if varIn.ndim == 1:
            climVar = rollavg_1Dconvolve(climVar, conv, wrap=True, square=False)[0]
            errVar  = rollavg_1Dconvolve(errVar,  conv, wrap=True, square=False)[0]
        else:
            # Tile, smooth along doy axis for each spatial/level point, untile
            # Uses moveaxis so it works for any number of spatial dimensions
            tLen    = n_doy
            climVar = np.concatenate([climVar]*3, axis=0)   # tile along doy
            errVar  = np.concatenate([errVar]*3,  axis=0)
            for idx in np.ndindex(spatial_shape):
                climVar[(slice(None),)+idx] = rollavg_1Dconvolve(
                    climVar[(slice(None),)+idx], conv, wrap=False, square=False)[0]
                errVar[(slice(None),)+idx]  = rollavg_1Dconvolve(
                    errVar[(slice(None),)+idx],  conv, wrap=False, square=False)[0]
            climVar = climVar[tLen:-tLen]             # untile
            errVar  = errVar[tLen:-tLen]

    # --- Compute anomalies ---
    for ii in range(np.min(np.unique(doyClim)) - 1, n_doy):
        idx = np.flatnonzero(doy == ii + 1)
        if len(idx) == 0:
            continue
        varAnom[idx] = varIn[idx] - climVar[ii]

    # --- Catch leap days (doy not in climatology) ---
    # Find time steps where anomaly is still NaN despite valid input data
    nan_mask = np.isnan(varAnom)
    if varIn.ndim > 1:
        nan_mask = nan_mask.any(axis=tuple(range(1, varIn.ndim)))
    leap_idx = np.flatnonzero(nan_mask & np.isfinite(varIn).any(
        axis=tuple(range(1, varIn.ndim))) if varIn.ndim > 1 
        else nan_mask & np.isfinite(varIn))
    for ii in leap_idx:
        d = doy[ii] - 1                              # 0-based index into climVar
        varAnom[ii] = varIn[ii] - np.nanmean(
            climVar[max(0, d-1):d+2], axis=0)        # average neighboring doys

    # --- Optional daily averaging (period mode only) ---
    if ops is not None and period is not None:
        varDaily     = make_daily(varIn[:nUse], ops=ops, pad=False)[0]
        varAnomDaily = make_daily(varAnom[:nUse], ops=ops, pad=False)[0]
        #varDaily     = varIn[:nUse].reshape((nUse // ops, ops) + spatial_shape).mean(axis=1).squeeze()
        return varAnom, climVar, errVar, varDaily, varAnomDaily

    return varAnom, climVar, errVar

def lomb_scargle_periodogram(t, y, freqs=None, norm=True):
    """
    Lomb-Scargle periodogram for unevenly sampled time series.

    Parameters
    ----------
    t     : time array (sols)
    y     : signal array (may contain NaN; NaN points are dropped)
    freqs : frequency array (cycles/sol).  If None, auto-determined.
    norm  : normalise power by variance of signal

    Returns
    -------
    freqs  : frequency array (cycles/sol)
    power  : Lomb-Scargle power
    periods: 1/freqs
    """
    mask = np.isfinite(y) & np.isfinite(t)
    t_clean = t[mask].astype(float)
    y_clean = y[mask].astype(float) - np.nanmean(y[mask])

    if freqs is None:
        T = t_clean[-1] - t_clean[0]
        dt_min = np.median(np.diff(np.sort(t_clean)))
        freq_min = 1.0 / T
        freq_max = 0.5 / dt_min
        freqs = np.linspace(freq_min, freq_max, 10000)

    power = sp.signal.lombscargle(t_clean, y_clean, 2 * np.pi * freqs,
                                   normalize=norm)
    return freqs, power, 1.0 / freqs


def morlet_wavelet(t, y, freqs=None, omega0=6.0):
    """
    Continuous Wavelet Transform (CWT) using a Morlet wavelet.

    Parameters
    ----------
    t      : evenly-spaced time array (sols); NaN points set to zero.
    y      : signal array (NaN → 0 before transform)
    freqs  : array of frequencies (cycles/sol) to evaluate
    omega0 : Morlet wavelet parameter (≈ number of oscillations per wavelet)

    Returns
    -------
    periods : period array (sols) = 1/freqs
    power   : CWT power, shape (len(freqs), len(t))
    coi     : cone-of-influence (sols)
    """
    y_clean = np.copy(y).astype(float)
    y_clean[np.isnan(y_clean)] = 0.0

    dt = np.nanmean(np.diff(t))
    N  = len(y_clean)

    if freqs is None:
        s_min = 2 * dt
        s_max = N * dt / 4
        nscales = 64
        freqs = 1.0 / np.geomspace(s_min, s_max, nscales)

    scales = omega0 / (2 * np.pi * freqs)   # Morlet scale ↔ frequency

    # Pad to next power of 2 for speed
    Npad = int(2 ** np.ceil(np.log2(N)))
    yf = np.fft.fft(y_clean, n=Npad)
    freqs_fft = np.fft.fftfreq(Npad, d=dt)
    omega = 2 * np.pi * freqs_fft

    power = np.zeros((len(scales), N))
    for j, s in enumerate(scales):
        # Morlet wavelet in frequency domain
        psi_hat = (np.pi ** (-0.25) *
                   np.exp(-0.5 * (s * omega - omega0) ** 2) *
                   np.sqrt(2 * np.pi * s / dt))
        W = np.fft.ifft(yf * psi_hat)[:N]
        power[j] = np.abs(W) ** 2

    # Cone of influence (e-folding time for Morlet)
    coi = np.sqrt(2) * scales[-1] * np.array(
        [min(i + 1, N - i) for i in range(N)]) * dt

    periods = 1.0 / freqs
    return periods, power, coi

def lomb_scargle_power_single_frequency_astropy( t, y, f, dy=None, center_data=True,
    fit_mean=True, normalization="standard", method="auto",):
    """
    Normalized Lomb–Scargle power at one frequency using astropy.timeseries.LombScargle.

    Parameters
    ----------
    t : array_like, shape (T,)
        Times.
    y : array_like, shape (T,)
        Observations.
    f : float
        Frequency (cycles per unit time).
    dy : array_like, shape (T,), optional
        1-sigma uncertainties of y. If provided, Astropy uses weighted LS.
        (If you instead have weights w=1/sigma^2, set dy = 1/sqrt(w).)
    center_data : bool
        If True, subtract the weighted mean of the data internally.
    fit_mean : bool
        If True, include a floating mean term in the model.
    normalization : str
        Astropy normalization ("standard", "model", "log", "psd").
    method : str
        Astropy method ("auto", "fast", "slow", "chi2", ...).

    Returns
    -------
    power : float
        Lomb–Scargle power at frequency f.
    """
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)

    ls = LombScargle(t, y, dy=dy, center_data=center_data, fit_mean=fit_mean)
    power = ls.power(f, normalization=normalization, method=method)
    return float(np.asarray(power))

def lomb_scargle_periodogram(t, y, fgrid, dy=None, center_data=True,
    fit_mean=True, normalization="standard", method="auto"):
    """
    Lomb–Scargle power over a frequency grid using Astropy.

    Parameters
    ----------
    t : array_like, shape (T,)
    y : array_like, shape (T,)
    fgrid : array_like, shape (M,)
        Frequencies (cycles per unit time).
    dy : array_like, shape (T,), optional
    center_data, fit_mean, normalization, method : see single-frequency function

    Returns
    -------
    power : ndarray, shape (M,)
    """
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    fgrid = np.asarray(fgrid, dtype=float)

    ls = LombScargle(t, y, dy=dy, center_data=center_data, fit_mean=fit_mean)
    power = ls.power(fgrid, normalization=normalization, method=method)
    return np.asarray(power, dtype=float)

In [ ]:
#load data
reanalysis="OMARS"
npzfile=np.load("/Users/battalio/Mars/"+reanalysis+"/MCS_U_daily.npz")
ops=1 #daily
uZonal=np.nanmean((npzfile['u'])[:,:,:,:],axis=-1)
lons=np.round(np.squeeze(npzfile['lons']),2)
lats=np.round(np.squeeze(npzfile['lats']),2)
plevs=np.squeeze(npzfile['plevs'])[:]
MYs=np.squeeze(npzfile['MY'])[:]
lss=np.squeeze(npzfile['lss'])[:]
MYlss=MYs+lss/360
sols=(np.round(np.squeeze(npzfile['sols'])[:])).astype(int)
#soy=(sols-sols[0]+27)%669+1

npzfile=np.load("/Users/battalio/Mars/"+reanalysis+"/MCS_EKEdel_daily.npz")
EKEZonal=np.nanmean((npzfile['EKE'])[:],axis=-1)
npzfile=np.load("/Users/battalio/Mars/"+reanalysis+"/MCS_UVdel_daily.npz")
momFluxZonal=np.nanmean((npzfile['uv'])[:],axis=-1)
npzfile=np.load("/Users/battalio/Mars/"+reanalysis+"/MCS_VTdel_daily.npz")
heatFluxZonal=np.nanmean((npzfile['vT'])[:],axis=-1)

idxTime=np.argwhere((MYlss>29) & (MYs<36))[:,0]
MYlss=MYlss[idxTime]
lss=lss[idxTime]
sols=sols[idxTime]
MYs=MYs[idxTime]

soy=np.zeros_like(MYs)
for ii in np.unique(MYs):
    MYloc=np.argwhere((MYs==int(ii)))[:,0]
    soy[MYloc]=np.arange(len(MYloc))

EYday=MYs+soy/np.max(soy)
soy=soy.astype(int)

#get roughly where data are missing
dta = ncopen("/Users/battalio/Mars/OMARS/MCS_ret_coverage.nc",'r')
Cov = np.nansum(np.array(dta.variables['MCS_T_profiles'][:]).astype(np.float32),axis=1)
Cov=np.interp(np.arange(len(Cov)*2)/2,np.arange(len(Cov)),Cov<300)[idxTime]
Cov=Cov>.09
diff=(np.diff(Cov.astype(int)))
startT=np.where(diff>0)[0]
endT=np.where(diff<0)[0]
if startT[0]>endT[0]:startT=np.concatenate(([0],startT.squeeze()))
times=list(zip(MYlss[startT],MYlss[endT]))

momFluxZonal=np.moveaxis(momFluxZonal[idxTime,:,:],[0, 1, 2], [2, 0, 1])
heatFluxZonal=np.moveaxis(heatFluxZonal[idxTime,:,:],[0, 1, 2], [2, 0, 1])
EKEZonal=np.moveaxis(EKEZonal[idxTime,:,:],[0, 1, 2], [2, 0, 1])
uZonal=np.moveaxis(uZonal[idxTime,:,:],[0, 1, 2], [2, 0, 1])

sqrtLats=np.sqrt(np.abs(np.cos(lats*np.pi/180))).squeeze()
hozWeight=np.cos(lats*np.pi/180).squeeze()
verWeight=(plevs/np.nanmax(plevs))
levDiff=np.diff(np.append(plevs, 0)).astype(float)*(-1)
levDiff=np.sqrt(levDiff/np.nanmax(levDiff))
levDiff=np.sqrt(verWeight)

idx85N=(np.abs(lats - 85)).argmin()
idx80N=(np.abs(lats - 80)).argmin()
idx75N=(np.abs(lats - 75)).argmin()
idx25N=(np.abs(lats - 25)).argmin()+1
idx20N=(np.abs(lats - 20)).argmin()+1
idx20S=(np.abs(lats - (-20))).argmin()
idx25S=(np.abs(lats - (-25))).argmin()
idx75S=(np.abs(lats - (-75))).argmin()+1
idx85S=(np.abs(lats - (-85))).argmin()+1
lev1=3
lev2=17
lev1=5
lev2=16


In [ ]:
uZonalWeight=uZonal*sqrtLats[None,:,None]*levDiff[:,None,None]
EKEZonalWeight=EKEZonal*sqrtLats[None,:,None]*levDiff[:,None,None]

In [ ]:
idxVar=np.argwhere((MYlss>30.15)&(MYs!=34))[:,0]
conv=31
uZonalAnomWeightYr,uZonalClimWeight=calc_Anom(uZonalWeight,doy=soy,idxClim=idxVar,conv=True,z_max=0,median=True)
EKEZonalAnomWeightYr,_=calc_Anom(EKEZonalWeight,soy,idxVar,conv=True,z_max=0)

In [ ]:
psAnomSeries = pyleo.Series(time =  np.arange(len(psAnomDailyYr[:669])), value = , label = 'anomalous U PC',
                  time_name = 'day', value_name = 'Anomalous zonal-win',
                  time_unit = 'Soy',   value_unit = 'm/s', verbose=False)

In [ ]:
psd_lomb = psHourSeries.spectral()  # method='lomb-scargle' by default 
fig1, ax1 = psd_lomb.plot(label='lomb',zorder=10)

psd_wwz = psHourSeries.spectral(method='wwz')
fig2, ax2 = psd_wwz.plot(label='wwz',zorder=10)